# 05 — Customer & Seller Intelligence

Olist E-Commerce Python Analytics

Covers: Seller Performance (moved from 04 Section 4) and Customer Value (moved from 04 Section 6), plus new extension questions that justify this as a standalone notebook. Full write-up of every finding lives in `05_customer_and_seller_intelligence_log.md` — comments here are kept short.

## Setup

This notebook reloads the cleaned tables independently rather than inheriting from 04, since it now stands on its own.

In [2]:
import pandas as pd
import numpy as np

customers = pd.read_csv('../data/cleaned/olist_customers_dataset.csv', dtype={'customer_zip_code_prefix': str})
orders = pd.read_csv('../data/cleaned/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/cleaned/olist_order_items_dataset.csv')
order_payments = pd.read_csv('../data/cleaned/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('../data/cleaned/olist_order_reviews_dataset.csv')
order_level_reviews = pd.read_csv('../data/cleaned/order_level_reviews.csv')
products = pd.read_csv('../data/cleaned/olist_products_dataset.csv')
category_translation = pd.read_csv('../data/cleaned/product_category_name_translation.csv')
sellers = pd.read_csv('../data/cleaned/olist_sellers_dataset.csv', dtype={'seller_zip_code_prefix': str})
geo_lookup = pd.read_csv('../data/cleaned/geo_lookup.csv', dtype={'geolocation_zip_code_prefix': str})

# Rebuild purchaseMonth and orderOverTime — needed downstream, not
# inherited from 04's cells.
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['purchaseMonth'] = orders['order_purchase_timestamp'].dt.to_period('M')
orderOverTime = orders.groupby('purchaseMonth')['order_id'].count()


## Section A — Seller Performance

In [3]:
sellers.columns
products.columns
order_items.columns
orders.columns


Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'lateDelivery', 'validDateCheck', 'delivery_days',
       'delivery_delay_days', 'purchaseMonth'],
      dtype='str')

#### Base table — anchor on order_items

In [4]:
# Anchored on order_items (one row per sale), sellers and products
# left-joined on — never the reverse, which would inflate counts with
# unsold catalog items.
sellersItemSold = order_items.merge(sellers, on='seller_id', how='left')
sellersItemPdtSold = sellersItemSold.merge(products, on='product_id', how='left')
sellerItemPdtSoldDelivery = sellersItemPdtSold.merge(orders, on='order_id', how='left')
# Finding: row count matches order_items exactly (112,650) — no catalog-side inflation.


#### Q1. Top sellers by order count vs. revenue

In [5]:
sellerOrderSale = sellersItemPdtSold.groupby('seller_id')['order_id'].nunique()
top30SellerOrderCnt = sellerOrderSale.sort_values(ascending=False).head(30)
print(top30SellerOrderCnt)

sellerRev = sellersItemPdtSold.groupby('seller_id')['item_total_value'].sum()
top30SellerRev = sellerRev.sort_values(ascending=False).head(30)
print(top30SellerRev)

overlap = set(top30SellerRev.index) & set(top30SellerOrderCnt.index)
print(len(overlap))
# Finding: ~15 of the top 20 overlap between the two rankings.


seller_id
6560211a19b47992c3666cc44a7e94c0    1854
4a3ca9315b744ce9f8e9374361493884    1806
cc419e0650a3c5ba77189a1882b7556a    1706
1f50f920176fa81dab994f9023523100    1404
da8622b14eb17ae2831f4ac5b9dab84a    1314
955fee9216a65b617aa5c0531780ce60    1287
7a67c85e85bb2ce8582c35f2203ad736    1160
ea8482cd71df3c1969d7b9473ff13abc    1146
4869f7a5dfa277a7dca6462dcf3b52b2    1132
3d871de0142ce09b7081e2b9d1733cb1    1080
7c67e1448b00f6e969d365cea6b010ab     982
8b321bb669392f5163d04c59e235e066     943
1025f0e2d44d7041d6cf58b6550e0bfa     915
620c87c171fb2a6dd6e8bb4dec959fc6     740
a1043bafd471dff536d0c462352beb48     718
cca3071e3e9bb7d12640c9fbe2301306     712
f8db351d8c4c4c22c6835c19a46f01b0     667
e9779976487b77c6d4ac45f75ec7afe9     657
fa1c13f2614d7b5c4749cbc52fecda94     585
7d13fca15225358621be4086e1eb0964     565
d91fb3b7d041e83b64a00a3edfb37e4f     535
128639473a139ac0f3e5f5ade55873a5     533
391fc6631aebcf3004804e51b40bcf1e     526
d2374cbcbb3ca4ab1086534108cc3ab7     524
46dc3b

#### Q2. Seller revenue concentration

In [6]:
totalGrossRevenue = sellersItemPdtSold['item_total_value'].sum()
for n in [5, 10, 30, 50, 100]:
    share = sellerRev.sort_values(ascending=False).head(n).sum() / totalGrossRevenue * 100
    print(n, share)
# Finding: seller revenue is sharply concentrated at the top cutoffs — steeper than category concentration (see 04 log, Q6).


5 7.350993128609577
10 12.804682126974695
30 25.224829300980723
50 32.243411516443395
100 44.37373386829986


#### Q3. Free-delivery orders vs. item value

In [7]:
print(sellersItemPdtSold[sellersItemPdtSold['freight_value'] == 0]['item_total_value'].mean() > sellersItemPdtSold['item_total_value'].mean())
print(sellersItemPdtSold['item_total_value'].mean() - sellersItemPdtSold[sellersItemPdtSold['freight_value'] == 0]['item_total_value'].mean())
# Finding: free-delivery orders average ~Rs42 LESS in item value — not a premium-order perk.


False
42.042570692977975


#### Q4. Delivery delay vs. review score

In [8]:
# Sign convention check: negative delay = delivered early
a = sellerItemPdtSoldDelivery.groupby('seller_id')['delivery_delay_days'].mean()
a[a > 0]
# Finding: negative = early is the norm for ~97% of sellers.


seller_id
08cdbae123ff67ca4e36d9d641ce0119     4.166667
1352e06ae67b410cdae0b2a22361167b     2.500000
13d95f0f6f73943d4ceffad0fc2cd32c     2.000000
154bdf805377afea75a3bd158e9eab10     2.000000
19484c79cef6c062cb177aa4ef2fcc3c     3.000000
                                      ...    
eebb3372362aa9a46975164bed19a7e7    27.000000
efb628aa07c8ca93d7db0d85b91a4f5a     1.000000
f22a247b79dd22ecd91dc8e3b2a1b07c     1.625000
f524ad65d7e0f1daab730ef2d2e86196     3.000000
f5fea3ffed6c2e889bab72705557c63a     1.000000
Name: delivery_delay_days, Length: 89, dtype: float64

In [9]:
sellerReviews = sellerItemPdtSoldDelivery.merge(
    order_level_reviews[['order_id', 'review_score_avg']],
    on='order_id',
    how='left'
)
sellerDeliveryReview = sellerReviews.groupby('seller_id').agg(
    avgDelay=('delivery_delay_days', 'mean'),
    avgReviewScore=('review_score_avg', 'mean')
)
print(sellerDeliveryReview.corr())
# Finding: r ~ -0.22 among sellers with delivered orders.


                avgDelay  avgReviewScore
avgDelay        1.000000       -0.223492
avgReviewScore -0.223492        1.000000


In [10]:
# Sellers with no delivered orders at all (NaN delay) — are they worse off?
sellerReviews[sellerReviews['delivery_delay_days'].isna()]['review_score_avg'].describe()
# Finding: mean review score 1.78, median 1.0 — non-delivery is a stronger driver of bad reviews than lateness alone.


count    2389.000000
mean        1.777103
std         1.335173
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         5.000000
Name: review_score_avg, dtype: float64

#### Q5. Seller size vs. review score and delivery delay

In [11]:
sellerRevenueReview = sellerReviews.groupby('seller_id').agg(
    totalRev=('item_total_value', 'sum'),
    avgReviewScore=('review_score_avg', 'mean')
)
print(sellerRevenueReview.corr())
# Finding: r ~ 0.02 — seller size is a non-factor for review score.


                totalRev  avgReviewScore
totalRev        1.000000        0.020684
avgReviewScore  0.020684        1.000000


In [12]:
sellerSizeDelivery = sellerReviews.groupby('seller_id').agg(
    totalRev=('item_total_value', 'sum'),
    avgDelay=('delivery_delay_days', 'mean')
)
print(sellerSizeDelivery.corr())
# Finding: r ~ 0.003 — seller size is also a non-factor for delivery delay.


          totalRev  avgDelay
totalRev  1.000000  0.002506
avgDelay  0.002506  1.000000


#### Q6-Q7. Average order value and category count per seller

In [13]:
sellerProfile = pd.DataFrame({
    'avgOrderValue': sellersItemPdtSold.groupby('seller_id')['item_total_value'].mean(),
    'categoryCount': sellersItemPdtSold.groupby('seller_id')['product_category_name'].nunique(),
    'totalRev': sellersItemPdtSold.groupby('seller_id')['item_total_value'].sum()
})
print(sellerProfile.sort_values('totalRev', ascending=False).head(20))
# Finding: top-10 sellers split into narrow high-ticket specialists vs. broad high-volume generalists.


                                  avgOrderValue  categoryCount   totalRev
seller_id                                                                
4869f7a5dfa277a7dca6462dcf3b52b2     215.952163             10  249640.70
7c67e1448b00f6e969d365cea6b010ab     175.613226              6  239536.44
53243585a1d6dc2643021fd1853d8905     575.260195              2  235856.68
4a3ca9315b744ce9f8e9374361493884     118.540493              7  235539.96
fa1c13f2614d7b5c4749cbc52fecda94     348.267457              5  204084.73
da8622b14eb17ae2831f4ac5b9dab84a     119.401883              4  185192.32
7e93a43ef30c4f03f38b393420bc753a     537.511912              8  182754.05
1025f0e2d44d7041d6cf58b6550e0bfa     121.050903              4  172860.69
7a67c85e85bb2ce8582c35f2203ad736     138.896994              2  162648.38
955fee9216a65b617aa5c0531780ce60     107.139880             23  160602.68
6560211a19b47992c3666cc44a7e94c0      74.405199              7  151265.77
1f50f920176fa81dab994f9023523100      

#### Q9. Mean vs. median order value per seller

In [14]:
sellerProfile = pd.DataFrame({
    'avgOrderValue': sellersItemPdtSold.groupby('seller_id')['item_total_value'].mean(),
    'medianOrderValue': sellersItemPdtSold.groupby('seller_id')['item_total_value'].median(),
    'totalRev': sellersItemPdtSold.groupby('seller_id')['item_total_value'].sum()
})
print(sellerProfile.sort_values('totalRev', ascending=False).head(20))
# Finding: some top sellers' mean is nearly double their median — outlier-driven, not typical.


                                  avgOrderValue  medianOrderValue   totalRev
seller_id                                                                   
4869f7a5dfa277a7dca6462dcf3b52b2     215.952163           190.980  249640.70
7c67e1448b00f6e969d365cea6b010ab     175.613226           168.935  239536.44
53243585a1d6dc2643021fd1853d8905     575.260195           288.710  235856.68
4a3ca9315b744ce9f8e9374361493884     118.540493           104.700  235539.96
fa1c13f2614d7b5c4749cbc52fecda94     348.267457           249.630  204084.73
da8622b14eb17ae2831f4ac5b9dab84a     119.401883           112.610  185192.32
7e93a43ef30c4f03f38b393420bc753a     537.511912           414.575  182754.05
1025f0e2d44d7041d6cf58b6550e0bfa     121.050903            80.865  172860.69
7a67c85e85bb2ce8582c35f2203ad736     138.896994           132.030  162648.38
955fee9216a65b617aa5c0531780ce60     107.139880            85.110  160602.68
6560211a19b47992c3666cc44a7e94c0      74.405199            64.230  151265.77

#### Q10. Monthly revenue trend for top sellers

In [15]:
top10SellerIds = sellerProfile.sort_values('totalRev', ascending=False).head(10).index
filtered = sellerItemPdtSoldDelivery[sellerItemPdtSoldDelivery['seller_id'].isin(top10SellerIds)]
sellerMonthlyRevenue = filtered.groupby(['purchaseMonth', 'seller_id'])['item_total_value'].sum().unstack()
print(sellerMonthlyRevenue)
# Finding: 2 of the top 10 peaked early and declined sharply; others are still growing — top-by-total-revenue is not top-currently.


seller_id      1025f0e2d44d7041d6cf58b6550e0bfa  \
purchaseMonth                                     
2016-10                                     NaN   
2017-01                                     NaN   
2017-02                                     NaN   
2017-03                                     NaN   
2017-04                                     NaN   
2017-05                                     NaN   
2017-06                                     NaN   
2017-07                                 1540.89   
2017-08                                 4883.23   
2017-09                                 9199.88   
2017-10                                 8060.91   
2017-11                                16869.57   
2017-12                                 1713.09   
2018-01                                10713.66   
2018-02                                 5286.59   
2018-03                                16561.53   
2018-04                                18023.28   
2018-05                        

#### Q_new3. Seller tenure — do newer sellers perform differently?

In [16]:
firstSaleMonth = sellerItemPdtSoldDelivery.groupby('seller_id')['purchaseMonth'].min()

sellerTenure = pd.DataFrame({'firstSaleMonth': firstSaleMonth})
# qcut needs a sortable numeric — a Period cast to str breaks chronological
# order, so use the Period's integer ordinal instead.
sellerTenure['firstSaleOrdinal'] = sellerTenure['firstSaleMonth'].astype('int64')
sellerTenure['tenureTier'] = pd.qcut(sellerTenure['firstSaleOrdinal'], q=3, labels=['Oldest', 'Mid', 'Newest'])
print(sellerTenure['tenureTier'].value_counts())
# Finding: tertiles reasonably balanced (Oldest 1128 / Mid 944 / Newest 1023).


tenureTier
Oldest    1128
Newest    1023
Mid        944
Name: count, dtype: int64


In [17]:
sellerTenurePerf = sellerReviews.groupby('seller_id').agg(
    totalRev=('item_total_value', 'sum'),
    avgReviewScore=('review_score_avg', 'mean'),
    avgDelay=('delivery_delay_days', 'mean')
)
sellerTenurePerf = sellerTenurePerf.merge(sellerTenure[['tenureTier']], on='seller_id', how='left')

tenureSummary = sellerTenurePerf.groupby('tenureTier').agg(
    avgRev=('totalRev', 'mean'),
    avgReviewScore=('avgReviewScore', 'mean'),
    avgDelay=('avgDelay', 'mean'),
    sellerCount=('totalRev', 'count')
)
print(tenureSummary)
# STATUS: tiers built and confirmed balanced above; this comparison has not yet been run/confirmed.


                 avgRev  avgReviewScore   avgDelay  sellerCount
tenureTier                                                     
Oldest      9035.461968        3.876755 -13.946975         1128
Mid         4399.826017        3.859340 -10.673760          944
Newest      1464.434389        4.183779 -11.929736         1023


## Section B — Customer Value

#### Q1. Repeat vs. one-time customer revenue share

In [18]:
uniqueCustomersOrderCount = customers.groupby('customer_unique_id')['customer_unique_id'].count()
repeatCustomers = uniqueCustomersOrderCount[uniqueCustomersOrderCount >= 2]

repeatCustomerOrders = customers[customers['customer_unique_id'].isin(repeatCustomers.index)].merge(
    orders, on='customer_id', how='left'
)
repeatCustomerOrdersItem = repeatCustomerOrders.merge(order_items, on='order_id', how='left')

singleTimeOrderCustomers = (customers[~(customers['customer_unique_id'].isin(repeatCustomers.index))]).merge(
    orders, on='customer_id', how='left'
)
singleTimeOrderCustomersItems = singleTimeOrderCustomers.merge(order_items, on='order_id', how='left')

print(repeatCustomerOrdersItem['item_total_value'].sum() / order_items['item_total_value'].sum() * 100)
print(singleTimeOrderCustomersItems['item_total_value'].sum() / order_items['item_total_value'].sum() * 100)
# Finding: repeat customers = 5.82% of revenue; one-time = 94.18%.


5.819669843202421
94.18033015679758


In [19]:
repeatCustomerCount = repeatCustomers.shape[0]
oneTimeCustomerCount = uniqueCustomersOrderCount[uniqueCustomersOrderCount == 1].shape[0]

print(repeatCustomerOrdersItem['item_total_value'].sum() / repeatCustomerCount)
print(singleTimeOrderCustomersItems['item_total_value'].sum() / oneTimeCustomerCount)
# Finding: avg revenue per repeat customer (~Rs308) is nearly double the one-time average (~Rs160), but there are too few repeat customers (~3%) to move total revenue.


307.65515181848514
160.27573604442583


#### Q2. RFM segmentation (Recency x Monetary)

In [20]:
customerOrderData = customers.merge(orders, on='customer_id', how='left')
customerOrderItemData = customerOrderData.merge(order_items, on='order_id', how='left')

a = customerOrderItemData.groupby('customer_unique_id').agg(
    totalSpend=('item_total_value', 'sum'),
    lastBuy=('order_purchase_timestamp', 'max')
)
a['lastBuy'] = pd.to_datetime(a['lastBuy']).dt.date
lastOrderDate = a['lastBuy'].max()
a['daySinceOrdered'] = lastOrderDate - a['lastBuy']
a = a.drop('lastBuy', axis=1)

a['monetaryTier'] = pd.qcut(a['totalSpend'], q=3, labels=['Low', 'Medium', 'High'])
a['recencyTier'] = pd.qcut(a['daySinceOrdered'], q=3, labels=['High', 'Medium', 'Low'])

rfmCrosstab = pd.crosstab(a['monetaryTier'], a['recencyTier'])
print(rfmCrosstab)
# Finding: near-uniform 3x3 crosstab — recency and monetary behave as independent variables here.


recencyTier    High  Medium    Low
monetaryTier                      
Low           10273   10925  10844
Medium        10771   10485  10767
High          11008   10760  10263


#### Q_new1. Customer acquisition trend

In [21]:
firstOrderMonth = customerOrderData.groupby('customer_unique_id')['purchaseMonth'].min()
newCustomersPerMonth = firstOrderMonth.groupby(firstOrderMonth.values).count()
print(newCustomersPerMonth)

orderToNewCustomerRatio = orderOverTime / newCustomersPerMonth
print(orderToNewCustomerRatio)
# Finding: ratio sits at ~1.0-1.04 nearly every month - volume is driven by new acquisition, not repeat buyers. (2018-09/10 spike is the same near-empty tail-off noise flagged in 04.)


2016-09       4
2016-10     321
2016-12       1
2017-01     764
2017-02    1752
2017-03    2636
2017-04    2352
2017-05    3596
2017-06    3139
2017-07    3894
2017-08    4184
2017-09    4130
2017-10    4470
2017-11    7304
2017-12    5487
2018-01    7025
2018-02    6451
2018-03    6965
2018-04    6711
2018-05    6622
2018-06    5940
2018-07    6071
2018-08    6271
2018-09       5
2018-10       1
Freq: M, Name: purchaseMonth, dtype: int64
purchaseMonth
2016-09    1.000000
2016-10    1.009346
2016-12    1.000000
2017-01    1.047120
2017-02    1.015982
2017-03    1.017451
2017-04    1.022109
2017-05    1.028921
2017-06    1.033769
2017-07    1.033898
2017-08    1.035134
2017-09    1.037530
2017-10    1.036018
2017-11    1.032859
2017-12    1.033898
2018-01    1.034733
2018-02    1.042939
2018-03    1.035319
2018-04    1.033974
2018-05    1.037904
2018-06    1.038215
2018-07    1.036403
2018-08    1.038431
2018-09    3.200000
2018-10    4.000000
Freq: M, dtype: float64


#### Q_new2 (Part 1). Customer geography — revenue by state

In [22]:
a = customerOrderItemData.groupby('customer_state').agg(
    totalRev=('item_total_value', 'sum'),
    uniqueCust=('customer_unique_id', 'nunique')
).sort_values('totalRev', ascending=False)
a['avgRev'] = a['totalRev'] / a['uniqueCust']
a['avgPrice'] = customerOrderItemData.groupby('customer_state')['price'].mean()
a['avgFreight'] = customerOrderItemData.groupby('customer_state')['freight_value'].mean()
print(a)
# Finding: SP has lowest avgRev/customer (Rs147) despite most revenue/customers; remote states highest (Rs240-272), driven by both higher freight AND higher price.


                  totalRev  uniqueCust      avgRev    avgPrice  avgFreight
customer_state                                                            
SP              5921678.12       40302  146.932612  109.653629   15.147275
RJ              2129681.98       12384  171.970444  125.117818   20.960924
MG              1856161.49       11259  164.860244  120.748574   20.630167
RS               885826.76        5277  167.865598  120.337453   21.735804
PR               800935.44        4882  164.058878  119.004139   20.531652
BA               611506.67        3277  186.605636  134.601208   26.363959
SC               610213.60        3534  172.669383  124.653578   21.470369
DF               353229.44        2075  170.231055  125.770549   21.041355
GO               347706.93        1952  178.128550  126.271732   22.766815
ES               324801.91        1964  165.377755  121.913701   22.058777
PE               322237.69        1609  200.272026  145.508322   32.917863
CE               275606.3

#### Q_new2 (Part 2). Customer geography — repeat rate by state

In [23]:
customerStateLookup = customers[['customer_unique_id', 'customer_state']].drop_duplicates('customer_unique_id')

orderCountWithState = uniqueCustomersOrderCount.rename('orderCount').reset_index()
orderCountWithState = orderCountWithState.merge(customerStateLookup, on='customer_unique_id', how='left')
orderCountWithState['isRepeat'] = orderCountWithState['orderCount'] >= 2

repeatRateByState = orderCountWithState.groupby('customer_state')['isRepeat'].mean() * 100
print(repeatRateByState.sort_values(ascending=False))
# Finding: no geographic pattern - high-volume states cluster at 3.0-3.4%; the extremes are small-sample noise.


customer_state
AC    5.194805
RO    4.184100
MT    3.428571
RJ    3.409550
GO    3.331625
AL    3.241895
SP    3.241097
RS    3.202577
MG    3.056419
PR    3.052028
DF    3.039074
SE    2.932551
ES    2.903719
PB    2.890173
BA    2.869353
SC    2.720317
MS    2.597403
TO    2.564103
PA    2.528978
MA    2.344828
RN    2.320675
PI    2.282158
RR    2.222222
PE    2.182045
AM    2.097902
CE    1.678108
AP    1.492537
Name: isRepeat, dtype: float64


In [25]:
# Export for 08's visualization notebook
rfmCrosstab.to_csv('../data/summary/rfm_crosstab.csv')
sellerRev.sort_values(ascending=False).to_csv('../data/summary/seller_revenue.csv', header=True)